In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, StackingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression

In [3]:
data_extended = pd.read_csv("data/on_time_performance.csv")

/tmp/ipykernel_740748/1105627841.py:1: DtypeWarning: Columns (0: Div2Airport, 1: Div2TailNum) have mixed types. Specify dtype option on import or set low_memory=False.
  data_extended = pd.read_csv("data/on_time_performance.csv")


In [8]:
columns_to_keep = ["Reporting_Airline", "Origin", "Dest", "FlightDate", "DayOfWeek", 
                   "Cancelled", "DepDelay", "ArrDelay", "CarrierDelay", "WeatherDelay", 
                   "NASDelay", "SecurityDelay", "LateAircraftDelay"]

data = data_extended.loc[:, columns_to_keep]

data["Delay"] = ((data["DepDelay"] > 0) | (data["ArrDelay"] > 0)).astype(int)
delay_columns = ["DepDelay", "ArrDelay", "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay", "Delay"]
data1 = data.drop(columns=delay_columns)
data1.dropna(inplace=True)

In [9]:
data1.info()

<class 'pandas.DataFrame'>
RangeIndex: 281266 entries, 0 to 281265
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Reporting_Airline  281266 non-null  str    
 1   Origin             281266 non-null  str    
 2   Dest               281266 non-null  str    
 3   FlightDate         281266 non-null  str    
 4   DayOfWeek          281266 non-null  int64  
 5   Cancelled          281266 non-null  float64
dtypes: float64(1), int64(1), str(4)
memory usage: 12.9 MB


In [10]:
y1 = data1["Cancelled"]
X = data1.drop(columns=["Cancelled"])
X_full, X_test, y_full, y_test = train_test_split(X, y1, test_size=0.25, random_state=5023)

X_train, X_val, y_train, y_val = train_test_split(X_full, y_full, test_size=0.25, random_state=5023)

nums_cols = X_train.select_dtypes(include = ["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include = ["object"]).columns

numeric_transformer = Pipeline([
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, nums_cols),
    ("cat", categorical_transformer, cat_cols)
])

# xgboost pipeline
xg_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", CatBoostClassifier(iterations=1000, learning_rate=0.1, depth=6))
])

xg_pipeline.fit(X_train, y_train)

# random forest
rf_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(n_estimators=100, random_state=6030))
])

rf_pipeline.fit(X_train, y_train)

# adaboost
ada_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", AdaBoostClassifier(n_estimators=100, random_state=6030))
])

ada_pipeline.fit(X_train, y_train)

# cat boost
cat_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", CatBoostClassifier(iterations=1000, learning_rate=0.1, depth=6))
])

cat_pipeline.fit(X_train, y_train)

# light gbm
lgb_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.1, max_depth=6))
])

lgb_pipeline.fit(X_train, y_train)

/tmp/ipykernel_740748/3888601325.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include = ["object"]).columns


0:	learn: 0.5376143	total: 57ms	remaining: 57s
1:	learn: 0.4266073	total: 66.5ms	remaining: 33.2s
2:	learn: 0.3467116	total: 79.4ms	remaining: 26.4s
3:	learn: 0.2891135	total: 88.3ms	remaining: 22s
4:	learn: 0.2468289	total: 97.3ms	remaining: 19.4s
5:	learn: 0.2132059	total: 107ms	remaining: 17.7s
6:	learn: 0.1906456	total: 116ms	remaining: 16.4s
7:	learn: 0.1731184	total: 125ms	remaining: 15.6s
8:	learn: 0.1585889	total: 135ms	remaining: 14.9s
9:	learn: 0.1469897	total: 145ms	remaining: 14.3s
10:	learn: 0.1396298	total: 154ms	remaining: 13.9s
11:	learn: 0.1326305	total: 164ms	remaining: 13.5s
12:	learn: 0.1273158	total: 173ms	remaining: 13.1s
13:	learn: 0.1225702	total: 183ms	remaining: 12.9s
14:	learn: 0.1187857	total: 192ms	remaining: 12.6s
15:	learn: 0.1158323	total: 201ms	remaining: 12.4s
16:	learn: 0.1131793	total: 211ms	remaining: 12.2s
17:	learn: 0.1107785	total: 220ms	remaining: 12s
18:	learn: 0.1089367	total: 230ms	remaining: 11.9s
19:	learn: 0.1071850	total: 239ms	remaining:

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

In [13]:
stack_model = StackingClassifier(
    estimators=[
        ("rf", rf_pipeline),
        ("ada", ada_pipeline),
        ("xgboost", xg_pipeline),
        ("lgb", lgb_pipeline),
        ("cat", cat_pipeline),
    ],
    final_estimator=LogisticRegression(max_iter=10000),
    cv=5,
    n_jobs=-1
)
stack_model.fit(X_train, y_train)


[LightGBM] [Info] Number of positive: 4677, number of negative: 153534
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.045458 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1220
[LightGBM] [Info] Number of data points in the train set: 158211, number of used features: 607
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.029562 -> initscore=-3.491265
[LightGBM] [Info] Start training from score -3.491265
0:	learn: 0.5376143	total: 88.3ms	remaining: 1m 28s
1:	learn: 0.4266073	total: 114ms	remaining: 57s
2:	learn: 0.3467116	total: 129ms	remaining: 42.9s
0:	learn: 0.5376143	total: 141ms	remaining: 2m 20s
3:	learn: 0.2891135	total: 152ms	remaining: 37.8s
4:	learn: 0.2468289	total: 173ms	remaining: 34.5s
1:	learn: 0.4266073	total: 187ms	remaining: 1m 33s
5:	learn: 0.2132059	total: 199ms	remaining: 33s
2:	learn: 0.3467116	total: 214ms	remaining: 1m 11s
6:	learn: 0.1906456	total: 220ms	remaining: 31.2s
7:	

/home/xxn2pb/.venv/lib64/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
112:	learn: 0.0766353	total: 15s	remaining: 1m 57s
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
78:	learn: 0.0810456	total: 13.5s	remaining: 2m 37s
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

/home/xxn2pb/.venv/lib64/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


84:	learn: 0.0800411	total: 14.5s	remaining: 2m 35s
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
117:	learn: 0.0761317	total: 16s	remaining: 1m 59s
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
84:	learn: 0.0805642	total: 15.9s	remaining: 2m 51s
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
89:	learn: 0.0801247	total: 14.7s	remaining: 2m 28s
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
90:	learn: 0.0799827	total: 14.7s	remaining: 2m 27s
[LightGBM] [Warning] No further splits with positive gain, best ga

/home/xxn2pb/.venv/lib64/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/xxn2pb/.venv/lib64/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


125:	learn: 0.0751468	total: 18.4s	remaining: 2m 7s
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
98:	learn: 0.0783501	total: 16.6s	remaining: 2m 30s
96:	learn: 0.0783113	total: 16.7s	remaining: 2m 35s
140:	learn: 0.0744628	total: 18.4s	remaining: 1m 52s
137:	learn: 0.0742203	total: 18s	remaining: 1m 52s
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
138:	learn: 0.0741516	total: 18.1s	remaining: 1m 51s
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
100:	learn: 0.0785664	total: 16.7s	remaining: 2m 28s
86:	learn: 0.0806094	total: 15.7s	remaining: 2m 44s
113:	learn: 0.0767075	total: 18.7s	remaining: 2m 25s
99:	learn: 0.0782434	total: 16.7s	remaining: 2m 30s
98:	learn: 0.0787299	total: 18.1s	remaining: 2m 44s
126:	learn: 0.0750570	total: 18.6s	remaining: 2m 7

/home/xxn2pb/.venv/lib64/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


131:	learn: 0.0745612	total: 19.2s	remaining: 2m 6s
144:	learn: 0.0736311	total: 18.8s	remaining: 1m 50s
145:	learn: 0.0741374	total: 19.2s	remaining: 1m 52s
105:	learn: 0.0779385	total: 17.4s	remaining: 2m 27s
102:	learn: 0.0776664	total: 17.5s	remaining: 2m 32s
92:	learn: 0.0798633	total: 16.4s	remaining: 2m 40s
104:	learn: 0.0778682	total: 18.8s	remaining: 2m 40s
145:	learn: 0.0735270	total: 18.9s	remaining: 1m 50s
117:	learn: 0.0763513	total: 19.6s	remaining: 2m 26s
132:	learn: 0.0744860	total: 19.4s	remaining: 2m 6s
105:	learn: 0.0777411	total: 18.8s	remaining: 2m 38s
103:	learn: 0.0778269	total: 17.5s	remaining: 2m 31s
106:	learn: 0.0778072	total: 17.5s	remaining: 2m 26s
93:	learn: 0.0797557	total: 16.6s	remaining: 2m 39s
103:	learn: 0.0775349	total: 17.6s	remaining: 2m 31s
146:	learn: 0.0740822	total: 19.4s	remaining: 1m 52s
134:	learn: 0.0758286	total: 19.5s	remaining: 2m 4s
146:	learn: 0.0734586	total: 19s	remaining: 1m 50s
104:	learn: 0.0776847	total: 17.7s	remaining: 2m 30s


,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.The type of estimator is generally expected to be a classifier.However, one can pass a regressor for some use case (e.g. ordinalregression).","[('rf', ...), ('ada', ...), ...]"
,"final_estimator final_estimator: estimator, default=NoneA classifier which will be used to combine the base estimators.The default classifier is a:class:`~sklearn.linear_model.LogisticRegression`.",LogisticRegre...ax_iter=10000)
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",5
,"stack_method stack_method: {'auto', 'predict_proba', 'decision_function', 'predict'}, default='auto'Methods called for each base estimator. It can be:* if 'auto', it will try to invoke, for each estimator, `'predict_proba'`, `'decision_function'` or `'predict'` in that order.* otherwise, one of `'predict_proba'`, `'decision_function'` or `'predict'`. If the method is not implemented by the estimator, it will raise an error.",'auto'
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary ` for more details.",-1
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by 

In [17]:
from sklearn.metrics import f1_score
models = {
    "Random Forest": rf_pipeline,
    "AdaBoost": ada_pipeline,
    "XGBoost": xg_pipeline,
    "LightGBM": lgb_pipeline,
    "CatBoost": cat_pipeline,
    "Stacking Model": stack_model
}

for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    print(f"\n{name}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")



Random Forest
Accuracy: 0.9755
F1 Score: 0.5169

AdaBoost
Accuracy: 0.9695
F1 Score: 0.0000

XGBoost
Accuracy: 0.9799
F1 Score: 0.5693


/home/xxn2pb/.venv/lib64/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



LightGBM
Accuracy: 0.9794
F1 Score: 0.5542

CatBoost
Accuracy: 0.9799
F1 Score: 0.5693


/home/xxn2pb/.venv/lib64/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Stacking Model
Accuracy: 0.9799
F1 Score: 0.5782
